In [17]:
from rdflib import Graph, Namespace, RDF, RDFS, Literal
from pint import UnitRegistry

# Initialize Pint
ureg = UnitRegistry()

# --- Step 1: Load QUDT ontology ---
graph = Graph()
graph.parse("https://qudt.org/2.1/vocab/unit", format="turtle")

# --- Step 2: Define namespaces ---
QUDT = Namespace("http://qudt.org/schema/qudt/")
UNIT = Namespace("http://qudt.org/vocab/unit/")
DCTERMS = Namespace("http://purl.org/dc/terms/")

# --- Step 3: Extract QUDT unit labels ---
mapping = {}
collisions = []
not_found = 0
total_units = 0
unmatched_units = []

for unit_uri in graph.subjects(RDF.type, QUDT.Unit):
    total_units += 1
    success = False

    # Look for English labels only
    labels = [
        str(label).lower()
        for label in graph.objects(unit_uri, RDFS.label)
        if isinstance(label, Literal)
        and (label.language in ("en", "en-US"))
    ]

    for label_str in labels:
        try:
            pint_unit = ureg.parse_units(label_str)
            pint_key = pint_unit  # Use Pint unit as dictionary key – more stable than string!
            if pint_key in mapping:
                collisions.append((str(pint_key), mapping[pint_key], str(unit_uri)))
            else:
                mapping[pint_key] = str(unit_uri)
            success = True
            break
        except Exception:
            continue

    if not success:
        unmatched_units.append(unit_uri)

# --- Step 4: Try matching by qudt:symbol for unmatched units ---
for unit_uri in unmatched_units:
    symbols = [
        str(symbol).lower()
        for symbol in graph.objects(unit_uri, QUDT.symbol)
        if isinstance(symbol, Literal)
    ]

    success = False
    for symbol_str in symbols:
        try:
            pint_unit = ureg.parse_units(symbol_str)
            pint_key = pint_unit
            if pint_key in mapping:
                collisions.append((str(pint_key), mapping[pint_key], str(unit_uri)))
            else:
                mapping[pint_key] = str(unit_uri)
            success = True
            break
        except Exception:
            continue

    if not success:
        not_found += 1

# --- Step 5: Print summary ---
print(f"\n✅ Parsed {len(mapping)} QUDT → Pint unit mappings successfully.")
print(f"⚠️ {len(collisions)} collisions detected (QUDT units with same Pint key).")
print(f"❌ {not_found} units could not be parsed by Pint.")
print(f"📦 Total QUDT units processed: {total_units}")

# --- Step 6: Show collisions ---
print("\n🔁 Collisions (same Pint unit key used twice):")
for pint_key, first, second in collisions[:10]:  # Show only first 10
    print(f"Pint '{pint_key}' → {first}  AND  {second}")

# --- Step 7: Show a few mappings ---
print("\n🔧 Example mappings:")
for pint_key, qudt_uri in list(mapping.items())[:10]:
    print(f"{str(pint_key):30} → {qudt_uri}")


final_mapping = {}

for pint_unit, qudt_uri in mapping.items():
    class_name = qudt_uri.split("/")[-1]  # extract last part of URI
    final_mapping[str(pint_unit)] = class_name

# Replace mapping with final_mapping if needed
mapping = final_mapping

# Show a few results
print("\n✅ Final mapping (Pint → QUDT class name):")
for pint_str, qudt_class in list(mapping.items())[:10]:
    print(f"{pint_str:50}  →  {qudt_class}")



✅ Parsed 1779 QUDT → Pint unit mappings successfully.
⚠️ 123 collisions detected (QUDT units with same Pint key).
❌ 673 units could not be parsed by Pint.
📦 Total QUDT units processed: 2575

🔁 Collisions (same Pint unit key used twice):
Pint 'barye' → http://qudt.org/vocab/unit/BARAD  AND  http://qudt.org/vocab/unit/BARYE
Pint 'dimensionless' → http://qudt.org/vocab/unit/BAR-PER-BAR  AND  http://qudt.org/vocab/unit/CentiM3-PER-CentiM3
Pint 'curie' → http://qudt.org/vocab/unit/CI  AND  http://qudt.org/vocab/unit/Ci
Pint 'erg / gram' → http://qudt.org/vocab/unit/ERG-PER-G  AND  http://qudt.org/vocab/unit/ERG-PER-GM
Pint 'franklin' → http://qudt.org/vocab/unit/C_Stat  AND  http://qudt.org/vocab/unit/FR
Pint 'dimensionless' → http://qudt.org/vocab/unit/BAR-PER-BAR  AND  http://qudt.org/vocab/unit/GM-PER-GM
Pint 'grade' → http://qudt.org/vocab/unit/GON  AND  http://qudt.org/vocab/unit/GR
Pint 'grade' → http://qudt.org/vocab/unit/GON  AND  http://qudt.org/vocab/unit/GRAD
Pint 'dimensionless

In [19]:
import json

with open("../../src/snakemake_report_plugin_metadata4ing/ontologies/qudt-mapping.json", "w") as f:
    json.dump(mapping, f, indent=4)